# htmlsight — Treinamento no Google Colab

Detector visual multi-task de componentes web Bootstrap.

**Antes de começar:**
1. Vá em `Ambiente de execução → Alterar tipo de ambiente de execução`
2. Selecione **GPU → T4** (gratuita) ou **A100** (Colab Pro)
3. Execute as células **em ordem, uma por vez**

---
> 🔴 **Se algo der errado:** rode a célula **"DIAGNÓSTICO COMPLETO"** logo abaixo e compartilhe o output.

## 🔍 DIAGNÓSTICO COMPLETO (rode a qualquer momento se algo der errado)

In [ ]:
# =====================================================================
# DIAGNÓSTICO COMPLETO — rode sempre que algo não funcionar
# Copie e cole TODO o output desta célula no chat para obter ajuda
# =====================================================================
import sys, os, subprocess
from pathlib import Path

SEP = '=' * 60

print(SEP)
print('SISTEMA')
print(SEP)
print(f'Python:       {sys.version}')
print(f'CWD:          {os.getcwd()}')
print(f'PYTHONPATH:   {os.environ.get("PYTHONPATH", "(não definido)")}')

print(f'\n{SEP}')
print('GPU')
print(SEP)
try:
    import torch
    print(f'torch:        {torch.__version__}')
    print(f'CUDA:         {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU:          {torch.cuda.get_device_name(0)}')
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        mem_free = torch.cuda.mem_get_info()[0] / 1e9
        print(f'VRAM total:   {mem:.1f} GB')
        print(f'VRAM livre:   {mem_free:.1f} GB')
except Exception as e:
    print(f'ERRO torch: {e}')

print(f'\n{SEP}')
print('PACOTES INSTALADOS')
print(SEP)
pacotes = ['ultralytics', 'playwright', 'PIL', 'typer', 'jinja2', 'faker', 'yaml']
for pkg in pacotes:
    try:
        mod = __import__(pkg if pkg != 'PIL' else 'PIL.Image', fromlist=['__version__'])
        ver = getattr(mod, '__version__', 'ok')
        print(f'  ✅ {pkg}: {ver}')
    except ImportError:
        print(f'  ❌ {pkg}: NÃO INSTALADO')

print(f'\n{SEP}')
print('REPOSITÓRIO')
print(SEP)
repo = Path('/content/htmlsight')
print(f'Existe /content/htmlsight: {repo.exists()}')
if repo.exists():
    r = subprocess.run(['git', 'log', '--oneline', '-3'], cwd=repo, capture_output=True, text=True)
    print(f'Últimos commits:\n{r.stdout}')
    src = repo / 'src/ia_visao_web'
    print(f'src/ia_visao_web existe: {src.exists()}')
    mods = list(src.glob('**/*.py')) if src.exists() else []
    print(f'Arquivos .py em src: {len(mods)}')

print(f'\n{SEP}')
print('IMPORT DA CLI')
print(SEP)
try:
    sys.path.insert(0, '/content/htmlsight/src')
    import ia_visao_web.cli as cli
    print(f'  ✅ ia_visao_web.cli importado com sucesso')
except Exception as e:
    print(f'  ❌ ERRO ao importar: {e}')

print(f'\n{SEP}')
print('DATASET')
print(SEP)
for base in ['/content/htmlsight', '/content/drive/MyDrive/htmlsight']:
    ds = Path(base) / 'data/dataset'
    if ds.exists():
        for split in ['train', 'val', 'test']:
            imgs = list((ds / 'images' / split).glob('*.png')) if (ds / 'images' / split).exists() else []
            lbls = list((ds / 'labels' / split).glob('*.txt')) if (ds / 'labels' / split).exists() else []
            print(f'  {base.split("/")[-1]}/{split}: {len(imgs)} imgs, {len(lbls)} labels')
        break
else:
    print('  Dataset não encontrado')

print(f'\n{SEP}')
print('PESOS DO MODELO (.pt)')
print(SEP)
for base in ['/content/htmlsight', '/content/drive/MyDrive/htmlsight', '/content']:
    pts = list(Path(base).rglob('*.pt')) if Path(base).exists() else []
    if pts:
        for pt in pts:
            size = pt.stat().st_size / 1e6
            print(f'  {pt}  ({size:.1f} MB)')
if not any(list(Path(b).rglob('*.pt')) for b in ['/content/htmlsight', '/content/drive/MyDrive/htmlsight', '/content'] if Path(b).exists()):
    print('  Nenhum arquivo .pt encontrado')

print(f'\n{SEP}')
print('LOGS DE TREINO')
print(SEP)
for base in ['/content/htmlsight/runs', '/content/drive/MyDrive/htmlsight/runs']:
    run_dir = Path(base)
    if run_dir.exists():
        csvs = list(run_dir.rglob('results.csv'))
        for csv in csvs:
            lines = csv.read_text().splitlines()
            print(f'  {csv} ({len(lines)-1} épocas registradas)')
            if len(lines) > 1:
                print(f'  Última linha: {lines[-1][:120]}')
        if not csvs:
            print(f'  Pasta runs existe mas sem results.csv (treino não iniciou ou travou antes de salvar)')
        break
else:
    print('  Pasta runs não encontrada (treino não foi executado)')

print(f'\n{SEP}')
print('PLAYWRIGHT / CHROMIUM')
print(SEP)
result = subprocess.run(['which', 'chromium-browser'], capture_output=True, text=True)
result2 = subprocess.run(['playwright', 'install', '--dry-run', 'chromium'], capture_output=True, text=True)
chromium_paths = list(Path('/root').rglob('chrome')) + list(Path('/usr').rglob('chromium*'))
print(f'  chromium-browser: {result.stdout.strip() or "não encontrado"}')
print(f'  chromium paths: {chromium_paths[:3] or "nenhum"}')

print(f'\n{SEP}')
print('FIM DO DIAGNÓSTICO — copie tudo acima e cole no chat')
print(SEP)

---
## 1. Verificar GPU

In [ ]:
import torch

print('=== GPU ===')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
    print(f'   CUDA version: {torch.version.cuda}')
else:
    print('⚠️  Sem GPU detectada!')
    print('   Vá em: Ambiente de execução → Alterar tipo → GPU → T4 → Salvar')
    print('   Depois reconecte e execute novamente do início.')

print(f'\ntorch: {torch.__version__}')

## 2. Montar Google Drive (opcional — persiste dataset e pesos)

⚠️ **Importante:** sem o Drive, tudo é apagado quando o Colab desconectar. O treino de 100 épocas leva ~2h — recomendado usar o Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/htmlsight')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Drive montado')
print(f'   Pasta: {DRIVE_DIR}')
print(f'   Conteúdo: {list(DRIVE_DIR.iterdir()) if DRIVE_DIR.exists() else "vazia"}')

## 3. Clonar repositório

In [ ]:
import os
from pathlib import Path

%cd /content

if Path('/content/htmlsight/.git').exists():
    print('Repositório já clonado. Atualizando...')
    !cd /content/htmlsight && git pull
else:
    print('Clonando repositório...')
    !git clone https://github.com/LucasMe110/htmlsight.git

%cd /content/htmlsight

import subprocess
log = subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True)
print(f'\n✅ Repositório pronto')
print(f'Branch: ', end='')
!git branch --show-current
print('Últimos commits:')
print(log.stdout)

## 4. Instalar dependências

In [ ]:
print('Instalando dependências do projeto...')
!pip install -q -e '.[dev,render,model]'
print('✅ Pacote base instalado')

print('\nInstalando Playwright...')
!pip install -q playwright
print('✅ Playwright instalado')

print('\nBaixando Chromium (necessário para gerar dataset real)...')
!playwright install chromium 2>&1 | tail -5
!playwright install-deps chromium 2>&1 | tail -3
print('✅ Chromium pronto')

In [ ]:
# Verificar TODOS os pacotes instalados
import sys
sys.path.insert(0, '/content/htmlsight/src')

print('=== VERIFICAÇÃO DE PACOTES ===')
checks = {
    'torch':        lambda: __import__('torch').__version__,
    'ultralytics':  lambda: __import__('ultralytics').__version__,
    'playwright':   lambda: __import__('playwright').__version__,
    'PIL':          lambda: __import__('PIL').__version__,
    'typer':        lambda: __import__('typer').__version__,
    'jinja2':       lambda: __import__('jinja2').__version__,
    'faker':        lambda: __import__('faker').__version__,
    'ia_visao_web': lambda: 'importado com sucesso',
}
all_ok = True
for name, fn in checks.items():
    try:
        ver = fn()
        print(f'  ✅ {name}: {ver}')
    except Exception as e:
        print(f'  ❌ {name}: FALHOU — {e}')
        all_ok = False

print()
if all_ok:
    print('✅ Todos os pacotes OK. Pode continuar.')
else:
    print('❌ Algum pacote falhou. Rode a célula de instalação novamente.')

# Rodar testes rápidos
print('\n=== TESTES RÁPIDOS ===')
import subprocess, os
r = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '--tb=short', '-x'],
    cwd='/content/htmlsight',
    env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
    capture_output=True, text=True
)
print(r.stdout[-2000:])  # últimas 2000 chars
if r.returncode == 0:
    print('✅ Todos os testes passaram')
else:
    print('❌ Testes falharam:')
    print(r.stderr[-1000:])

## 5. Configurar caminhos

Edite `USE_DRIVE = True` se montou o Google Drive na célula 2.

In [ ]:
import os, sys
from pathlib import Path

USE_DRIVE = False  # ← mude para True se montou o Google Drive

if USE_DRIVE:
    BASE = Path('/content/drive/MyDrive/htmlsight')
    if not BASE.exists():
        print('⚠️  Drive não montado! Execute a célula 2 primeiro.')
else:
    BASE = Path('/content/htmlsight')

DATASET_DIR = BASE / 'data/dataset'
RUNS_DIR    = BASE / 'runs'
REPORT_DIR  = RUNS_DIR / 'baseline/report'

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# Definir PYTHONPATH globalmente para todas as células seguintes
os.environ['PYTHONPATH'] = '/content/htmlsight/src'
sys.path.insert(0, '/content/htmlsight/src')

print('=== CAMINHOS CONFIGURADOS ===')
print(f'  BASE:        {BASE}  {"(existe)" if BASE.exists() else "(NÃO EXISTE)"}')
print(f'  DATASET_DIR: {DATASET_DIR}')
print(f'  RUNS_DIR:    {RUNS_DIR}')
print(f'  PYTHONPATH:  {os.environ["PYTHONPATH"]}')

# Verificar espaço em disco
import shutil
total, used, free = shutil.disk_usage('/')
print(f'\n  Disco livre: {free / 1e9:.1f} GB de {total / 1e9:.1f} GB total')
if free / 1e9 < 5:
    print('  ⚠️  Pouco espaço! Dataset de 3000 imgs ocupa ~1.5 GB.')

## 6. Gerar dataset

| Modo | Tempo | Qualidade |
|------|-------|-----------|
| `SYNTHETIC_ONLY=True` + `NUM_IMAGES=100` | ~30s | Só para testar o pipeline |
| `SYNTHETIC_ONLY=False` + `NUM_IMAGES=3000` | ~20min | Dataset real para treino |

In [ ]:
SYNTHETIC_ONLY = False  # True = rápido/sintético | False = real com Playwright
NUM_IMAGES     = 3000   # use 50-100 para testar pipeline
NUM_WORKERS    = 4

import subprocess, sys, os, time
from pathlib import Path

# Checar se já existe dataset
existing = list(DATASET_DIR.glob('images/**/*.png'))
if existing:
    print(f'⚠️  Já existem {len(existing)} imagens em {DATASET_DIR}')
    print('   Continuando com as imagens existentes. Para regenerar, delete a pasta data/dataset.')
else:
    cmd = [
        sys.executable, '-m', 'ia_visao_web.cli', 'dataset', 'build',
        '--count', str(NUM_IMAGES),
        '--workers', str(NUM_WORKERS),
        '--output', str(DATASET_DIR),
    ]
    if SYNTHETIC_ONLY:
        cmd.append('--synthetic-only')

    print(f'Gerando {NUM_IMAGES} imagens ({"sintético" if SYNTHETIC_ONLY else "Playwright real"})...')
    print(f'Comando: {" ".join(cmd)}')
    t0 = time.time()

    result = subprocess.run(
        cmd,
        env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
        cwd='/content/htmlsight'
    )

    elapsed = time.time() - t0
    print(f'\nTempo: {elapsed:.0f}s  |  exit code: {result.returncode}')

    if result.returncode != 0:
        print('❌ ERRO na geração do dataset. Rode a célula DIAGNÓSTICO COMPLETO.')

# Resumo do que foi gerado
print('\n=== RESUMO DO DATASET ===')
for split in ['train', 'val', 'test']:
    imgs  = list((DATASET_DIR / 'images' / split).glob('*.png')) if (DATASET_DIR / 'images' / split).exists() else []
    lbls  = list((DATASET_DIR / 'labels' / split).glob('*.txt')) if (DATASET_DIR / 'labels' / split).exists() else []
    attrs = list((DATASET_DIR / 'attrs'  / split).glob('*.json')) if (DATASET_DIR / 'attrs' / split).exists() else []
    status = '✅' if len(imgs) == len(lbls) == len(attrs) and len(imgs) > 0 else '❌'
    print(f'  {status} {split:6s}: {len(imgs):4d} imgs  {len(lbls):4d} labels  {len(attrs):4d} attrs')

yaml = DATASET_DIR / 'data.yaml'
print(f'  data.yaml: {"✅ existe" if yaml.exists() else "❌ NÃO ENCONTRADO"}')

total_imgs = list(DATASET_DIR.glob('images/**/*.png'))
print(f'\nTotal: {len(total_imgs)} imagens')

## 7. Validar dataset

In [ ]:
import subprocess, sys, os, json
from pathlib import Path
from IPython.display import Image as IPyImage, display

result = subprocess.run(
    [sys.executable, '-m', 'ia_visao_web.cli', 'dataset', 'validate',
     '--root', str(DATASET_DIR),
     '--report',
     '--min-train-instances', '10',
     '--qa-samples', '3'],
    env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
    cwd='/content/htmlsight',
    capture_output=True, text=True
)
print('STDOUT:', result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:1000])
print(f'Exit code: {result.returncode}')

# Distribuição de classes
report_path = DATASET_DIR / '_qa/report.json'
if report_path.exists():
    r = json.loads(report_path.read_text())
    counts = r.get('class_counts', {}).get('train', {})
    min_count = min(counts.values()) if counts else 0
    max_count = max(counts.values()) if counts else 0
    print(f'\n=== CLASSES NO TRAIN ({len(counts)} classes) ===')
    print(f'Min instâncias: {min_count}  |  Max: {max_count}')
    for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
        bar = '█' * min(25, n // 800)
        ok  = '✅' if n >= 10 else '⚠️ '
        print(f'  {ok} {cls:12s} {n:6d}  {bar}')

# Mostrar 1 imagem de QA
qa_imgs = list((DATASET_DIR / '_qa').glob('*.png'))
if qa_imgs:
    print(f'\n=== QA VISUAL (amostra) ===')
    display(IPyImage(filename=str(qa_imgs[0]), width=700))
else:
    print('Nenhuma imagem de QA gerada')

## 8. Treinar modelo

| GPU | Epochs | Tempo estimado |
|-----|--------|----------------|
| T4 (16 GB) | 100 | ~2–3 horas |
| A100 (40 GB) | 100 | ~45 min |
| CPU | 1 | ~30 min (só para testar) |

> 💡 Se der **OOM (out of memory)**: reduza `BATCH_SIZE` para 8 e reexecute.

In [ ]:
EPOCHS     = 100  # use 1 para smoke test rápido
BATCH_SIZE = 16   # reduza para 8 se der OOM
IMAGE_SIZE = 640

import subprocess, sys, os, time, torch
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'=== INICIANDO TREINO ===')
print(f'Device:     {device}')
print(f'Epochs:     {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Image size: {IMAGE_SIZE}')
print(f'Dataset:    {DATASET_DIR}')
print(f'Output:     {RUNS_DIR / "baseline"}')

if not (DATASET_DIR / 'data.yaml').exists():
    print('\n❌ data.yaml não encontrado! Execute a célula 6 (gerar dataset) primeiro.')
else:
    t0 = time.time()
    result = subprocess.run(
        [
            sys.executable, '-m', 'ia_visao_web.cli', 'train',
            '--dataset',    str(DATASET_DIR),
            '--output',     str(RUNS_DIR / 'baseline'),
            '--epochs',     str(EPOCHS),
            '--batch-size', str(BATCH_SIZE),
            '--image-size', str(IMAGE_SIZE),
            '--device',     device,
        ],
        env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
        cwd='/content/htmlsight',
    )
    elapsed = time.time() - t0
    h, m = divmod(int(elapsed), 3600)
    m, s = divmod(m, 60)
    print(f'\nTempo total: {h:02d}h {m:02d}m {s:02d}s  |  exit code: {result.returncode}')

    # Verificar resultado
    weights = RUNS_DIR / 'baseline/weights/best.pt'
    last_pt = RUNS_DIR / 'baseline/weights/last.pt'
    csv     = list((RUNS_DIR / 'baseline').rglob('results.csv'))

    print('\n=== RESULTADO DO TREINO ===')
    if weights.exists():
        size = weights.stat().st_size / 1e6
        print(f'  ✅ best.pt:  {weights}  ({size:.1f} MB)')
    else:
        print(f'  ❌ best.pt NÃO ENCONTRADO em {RUNS_DIR / "baseline/weights"}')

    if last_pt.exists():
        print(f'  ✅ last.pt:  {last_pt}')

    if csv:
        lines = csv[0].read_text().splitlines()
        print(f'  ✅ results.csv: {len(lines)-1} épocas')
        if len(lines) > 1:
            header = [h.strip() for h in lines[0].split(',')]
            last   = [v.strip() for v in lines[-1].split(',')]
            print('\n  Última época:')
            for h, v in zip(header, last):
                if any(k in h for k in ['map', 'loss', 'epoch']):
                    print(f'    {h}: {v}')
    else:
        print('  ⚠️  results.csv não encontrado (treino pode ter travado)')
        if result.returncode != 0:
            print('  ❌ Treino saiu com erro. Rode a célula DIAGNÓSTICO COMPLETO.')

## 9. Gerar relatório de métricas

In [ ]:
import subprocess, sys, os
from pathlib import Path
from IPython.display import Markdown, display

weights = RUNS_DIR / 'baseline/weights/best.pt'

if not weights.exists():
    print('⚠️  Pesos não encontrados. Execute a célula 8 (treinar) primeiro.')
    print(f'   Procurado em: {weights}')
    # Procurar em outros lugares
    pts = list(Path('/content').rglob('best.pt'))
    if pts:
        print(f'   Encontrado em: {pts[0]}')
        print(f'   Use: weights = Path("{pts[0]}")')
else:
    print(f'Gerando relatório com pesos: {weights}')
    result = subprocess.run(
        [
            sys.executable, '-m', 'ia_visao_web.cli', 'report',
            '--dataset', str(DATASET_DIR),
            '--weights', str(weights),
            '--output',  str(REPORT_DIR),
            '--split',   'test',
        ],
        env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
        cwd='/content/htmlsight',
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[:2000])

    md_path = REPORT_DIR / 'report.md'
    if md_path.exists():
        print('\n✅ Relatório gerado!')
        display(Markdown(md_path.read_text()))
    else:
        print(f'❌ report.md não encontrado em {REPORT_DIR}')

## 10. Predição em imagem

In [ ]:
import subprocess, sys, os, json
from pathlib import Path
from IPython.display import Image as IPyImage, display

weights = RUNS_DIR / 'baseline/weights/best.pt'

test_images = sorted((DATASET_DIR / 'images/test').glob('*.png')) if (DATASET_DIR / 'images/test').exists() else []
if not test_images:
    print('❌ Sem imagens no test set. Execute a célula 6 primeiro.')
else:
    img_path = test_images[0]
    print(f'Imagem: {img_path.name}')
    print(f'Pesos:  {weights if weights.exists() else "❌ não encontrado (retorna detections: [])"}\n')

    cmd = [sys.executable, '-m', 'ia_visao_web.cli', 'predict', str(img_path)]
    if weights.exists():
        cmd += ['--weights', str(weights)]

    result = subprocess.run(
        cmd,
        env={**os.environ, 'PYTHONPATH': '/content/htmlsight/src'},
        cwd='/content/htmlsight',
        capture_output=True, text=True
    )

    display(IPyImage(filename=str(img_path), width=700))

    print('=== DETECÇÕES ===')
    if result.returncode != 0:
        print(f'❌ Erro (exit {result.returncode}): {result.stderr[:500]}')
    else:
        try:
            data = json.loads(result.stdout)
            dets = data.get('detections', [])
            print(f'{len(dets)} detecções encontradas')
            for d in dets[:10]:
                print(f"  {d['class']:12s}  score={d['score']:.2f}  bbox={[round(x) for x in d['bbox']]}")
            if not dets:
                print('  (nenhuma) — modelo não treinado ou threshold muito alto')
        except Exception:
            print(result.stdout[:500])

## 11. Baixar pesos e relatório

Se não usou Google Drive, baixe os arquivos **antes de fechar o Colab**.

In [ ]:
from google.colab import files
from pathlib import Path

to_download = [
    RUNS_DIR / 'baseline/weights/best.pt',
    RUNS_DIR / 'baseline/weights/last.pt',
    REPORT_DIR / 'report.md',
    REPORT_DIR / 'report.json',
]

print('=== DOWNLOAD ===')
for f in to_download:
    p = Path(f)
    if p.exists():
        size = p.stat().st_size / 1e6
        print(f'Baixando {p.name} ({size:.1f} MB)...')
        files.download(str(p))
    else:
        print(f'⚠️  Não encontrado: {f}')